# SARSA y Q-learning sobre la escalera

**Módulo 23 — Aprendizaje por Refuerzo**

Este notebook implementa SARSA y Q-learning sobre el mismo problema de la escalera del módulo 21,
y extiende el análisis a un gridworld 4×4.

**Parámetros base:** α=0.5, γ=1, ε=0.4

## 1. Imports y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'blue':   '#2E86AB',
    'red':    '#E94F37',
    'green':  '#27AE60',
    'gray':   '#7F8C8D',
    'orange': '#F39C12',
    'dark':   '#2C3E50',
}
np.random.seed(42)

## 2. Ambiente: la escalera (módulo 21)

- 6 estados: s = 0, 1, 2, 3, 4, 5 (terminal)
- 2 acciones: subir 1 (+1) o saltar 2 (+2); desde s=4 solo +1
- Costos c = [3, 2, 5, 10, 1, 0]
- Recompensas r_i = -c_i (llegamos a s': recibimos R(s') = -c[s'])

In [ ]:
# ── Parámetros del ambiente ───────────────────────────────────────────────────
COSTS   = np.array([3, 2, 5, 10, 1, 0], dtype=float)
REWARDS = -COSTS
N_STATES = len(COSTS)   # 6: estados 0..5
GOAL     = N_STATES - 1  # estado 5
N_ACTIONS = 2            # 0=+1, 1=+2

def available_actions(s):
    """Índices de acciones disponibles desde s."""
    if s >= GOAL:
        return []
    if s == GOAL - 1:
        return [0]          # solo +1
    return [0, 1]

def step(s, a):
    """Ejecuta acción a desde estado s. Devuelve (s_next, r)."""
    s_next = s + (1 if a == 0 else 2)
    r = REWARDS[s_next]
    return s_next, r

# Verificación rápida
s_next, r = step(0, 1)  # desde s=0, acción +2
print(f"step(0, +2) → s'={s_next}, r={r}  (esperado: s'=2, r=-5)")

## 3. Política ε-greedy

In [ ]:
def eps_greedy(Q, s, eps, rng):
    """Selecciona acción ε-greedy desde estado s."""
    avail = available_actions(s)
    if len(avail) == 1:
        return avail[0]
    if rng.random() < eps:
        return int(rng.choice(avail))   # explorar
    q_vals = [Q[s, a] for a in avail]
    return avail[int(np.argmax(q_vals))]  # explotar

## 4. SARSA

**Regla de actualización:**

$$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma\, Q(s', a') - Q(s,a)\bigr]$$

donde $a' \sim \pi_\varepsilon(s')$ — la siguiente acción se elige **antes** de avanzar (on-policy).

In [ ]:
def sarsa(n_episodes=200, alpha=0.5, gamma=1.0, eps=0.4, seed=0,
          snapshot_episodes=None):
    """
    Ejecuta SARSA sobre la escalera.
    
    Returns:
        Q: tabla Q final (N_STATES x N_ACTIONS)
        episode_returns: lista de retornos por episodio
        snapshots: dict {episodio: Q_snapshot} si snapshot_episodes dado
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    Q[GOAL - 1, 1] = np.nan   # +2 no disponible desde s=4
    episode_returns = []
    snapshots = {}

    for ep in range(n_episodes):
        s = 0
        a = eps_greedy(Q, s, eps, rng)  # primer a elegida antes del bucle
        G = 0.0

        while s != GOAL:
            s2, r = step(s, a)
            G += r

            # Elige a' on-policy (SARSA)
            a2 = eps_greedy(Q, s2, eps, rng) if s2 != GOAL else 0

            # Target SARSA: Q(s', a')
            q_next = Q[s2, a2] if s2 != GOAL else 0.0
            delta = r + gamma * q_next - Q[s, a]
            Q[s, a] += alpha * delta

            s, a = s2, a2   # avanzamos con el 5-tuple completo

        episode_returns.append(G)

        if snapshot_episodes and (ep + 1) in snapshot_episodes:
            snapshots[ep + 1] = Q.copy()

    return Q, episode_returns, snapshots

## 5. Q-learning

**Regla de actualización:**

$$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[r + \gamma \max_{a'} Q(s', a') - Q(s,a)\bigr]$$

Diferencia con SARSA: el target usa $\max_{a'}$ en lugar de $Q(s', a')$ con $a' \sim \pi_\varepsilon$ (off-policy).

In [ ]:
def q_learning(n_episodes=200, alpha=0.5, gamma=1.0, eps=0.4, seed=0,
               snapshot_episodes=None):
    """
    Ejecuta Q-learning sobre la escalera.
    
    Returns:
        Q: tabla Q final (N_STATES x N_ACTIONS)
        episode_returns: lista de retornos por episodio
        snapshots: dict {episodio: Q_snapshot} si snapshot_episodes dado
    """
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    Q[GOAL - 1, 1] = np.nan
    episode_returns = []
    snapshots = {}

    for ep in range(n_episodes):
        s = 0
        G = 0.0

        while s != GOAL:
            a = eps_greedy(Q, s, eps, rng)
            s2, r = step(s, a)
            G += r

            # Target Q-learning: max_a' Q(s', a')
            avail2 = available_actions(s2)
            max_q_next = max(Q[s2, aa] for aa in avail2) if avail2 else 0.0
            delta = r + gamma * max_q_next - Q[s, a]
            Q[s, a] += alpha * delta

            s = s2   # avanzamos con solo el 4-tuple

        episode_returns.append(G)

        if snapshot_episodes and (ep + 1) in snapshot_episodes:
            snapshots[ep + 1] = Q.copy()

    return Q, episode_returns, snapshots

## 6. Visualización de la tabla Q

In [ ]:
def plot_q_table(Q, title, ax=None, opt_policy=None):
    """Dibuja la tabla Q como un heatmap anotado."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    
    Q_plot = Q[:GOAL].copy()  # estados 0..4
    Q_finite = np.where(np.isnan(Q_plot), 0, Q_plot)
    
    im = ax.imshow(Q_finite, cmap='RdYlGn', aspect='auto',
                   vmin=-12, vmax=2)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['+1 (subir)', '+2 (saltar)'])
    ax.set_yticks(range(GOAL))
    ax.set_yticklabels([f's={i}' for i in range(GOAL)])
    ax.set_title(title, fontsize=11)
    
    for i in range(GOAL):
        for j in range(N_ACTIONS):
            val = Q_plot[i, j]
            if np.isnan(val):
                ax.text(j, i, '—', ha='center', va='center', fontsize=12)
            else:
                marker = ''
                if opt_policy is not None and j == opt_policy[i]:
                    marker = '*'
                ax.text(j, i, f'{val:.2f}{marker}',
                        ha='center', va='center', fontsize=10,
                        color='black', fontweight='bold' if marker else 'normal')
    
    return ax

## 7. Ejecutar ambos algoritmos con los mismos parámetros

In [ ]:
# Parámetros base del módulo
ALPHA    = 0.5
GAMMA    = 1.0
EPSILON  = 0.4
N_EP     = 200
SEED     = 42
SNAPS    = [1, 2, 10, 50, 100, 200]

Q_sarsa,  ret_sarsa,  snaps_sarsa  = sarsa(
    n_episodes=N_EP, alpha=ALPHA, gamma=GAMMA,
    eps=EPSILON, seed=SEED, snapshot_episodes=SNAPS)

Q_ql, ret_ql, snaps_ql = q_learning(
    n_episodes=N_EP, alpha=ALPHA, gamma=GAMMA,
    eps=EPSILON, seed=SEED, snapshot_episodes=SNAPS)

print("Retorno promedio últimos 20 episodios:")
print(f"  SARSA:      {np.mean(ret_sarsa[-20:]):.2f}")
print(f"  Q-learning: {np.mean(ret_ql[-20:]):.2f}")
print(f"  Óptimo:     -6.00")

## 8. Curvas de aprendizaje

In [ ]:
def smooth(x, w=10):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(10, 4))
eps_axis = np.arange(1, N_EP + 1)

ax.plot(eps_axis, ret_sarsa, alpha=0.2, color=COLORS['blue'])
ax.plot(eps_axis[:len(smooth(ret_sarsa))], smooth(ret_sarsa),
        color=COLORS['blue'], lw=2, label='SARSA')

ax.plot(eps_axis, ret_ql, alpha=0.2, color=COLORS['orange'])
ax.plot(eps_axis[:len(smooth(ret_ql))], smooth(ret_ql),
        color=COLORS['orange'], lw=2, label='Q-learning')

ax.axhline(-6, color=COLORS['green'], ls='--', lw=1.5, label='Óptimo = −6')
ax.set_xlabel('Episodio')
ax.set_ylabel('Retorno')
ax.set_title(f'Curvas de aprendizaje (α={ALPHA}, γ={GAMMA}, ε={EPSILON})')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Evolución de la tabla Q

In [ ]:
show_eps = [1, 2, 50, 200]
fig, axes = plt.subplots(2, len(show_eps), figsize=(14, 7))

for col, ep in enumerate(show_eps):
    plot_q_table(snaps_sarsa[ep], f'SARSA — ep {ep}', ax=axes[0, col])
    plot_q_table(snaps_ql[ep],    f'Q-learning — ep {ep}', ax=axes[1, col])

fig.suptitle('Evolución de la tabla Q por episodio', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 10. Verificación: ¿Q* coincide con el módulo 21?

La política óptima del módulo 21 era: 0→(+2)→2→(+2)→4→(+1)→5, retorno = −6.

Extraemos `argmax_a Q*(s,a)` de Q-learning y verificamos.

In [ ]:
# Q* analítica del módulo 21 (γ=1)
Q_STAR_REF = np.array([
    [-8.0, -6.0],    # s=0  optimo: +2
    [-6.0, -10.0],   # s=1  optimo: +1
    [-10.0, -1.0],   # s=2  optimo: +2
    [-1.0,  0.0],    # s=3  optimo: +2
    [ 0.0,  np.nan], # s=4  optimo: +1
])
OPT_POLICY_REF = [1, 0, 1, 1, 0]  # índices de acción óptima
ACTION_NAMES = ['+1 (subir 1)', '+2 (saltar 2)']

print("Política extraída de Q-learning (argmax_a Q*(s,a)):")
print()

policy_ql = []
for s in range(GOAL):
    avail = available_actions(s)
    q_vals = [Q_ql[s, a] for a in avail]
    best_a = avail[int(np.argmax(q_vals))]
    policy_ql.append(best_a)
    ref = OPT_POLICY_REF[s]
    match = 'OK' if best_a == ref else 'DIFERENTE'
    print(f"  s={s}: Q-learning={ACTION_NAMES[best_a]:18s}  referencia={ACTION_NAMES[ref]}  [{match}]")

# Traza la trayectoria
print()
print("Trayectoria óptima según Q-learning:")
s, traj, ret = 0, [0], 0.0
while s != GOAL:
    a = policy_ql[s]
    s2, r = step(s, a)
    traj.append(s2)
    ret += r
    s = s2
print(f"  {' → '.join(map(str, traj))}  retorno = {ret:.1f}")
print(f"  Módulo 21: 0 → 2 → 4 → 5  retorno = -6.0")

## 11. Extensión: Gridworld 4×4

Ahora aplicamos los mismos algoritmos a un gridworld 4×4:
- Estados: celdas (fila, columna) = 0..15
- Acciones: arriba, abajo, izquierda, derecha
- Recompensas: −1 por paso, +10 al llegar a la meta (celda 15)
- Paredes: el agente no sale del grid (acción sin efecto)

```
┌────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │
├────┼────┼────┼────┤
│  4 │  5 │  6 │  7 │
├────┼────┼────┼────┤
│  8 │  9 │ 10 │ 11 │
├────┼────┼────┼────┤
│ 12 │ 13 │ 14 │ 15 │← meta
└────┴────┴────┴────┘
```

In [ ]:
# ── Ambiente gridworld ────────────────────────────────────────────────────────
GW_ROWS, GW_COLS = 4, 4
GW_N = GW_ROWS * GW_COLS   # 16 estados
GW_GOAL = GW_N - 1          # estado 15 = (3,3)
GW_ACTIONS = 4              # 0=arriba, 1=abajo, 2=izq, 3=der
GW_STEP_R  = -1.0
GW_GOAL_R  = +10.0

def gw_step(s, a):
    r, c = divmod(s, GW_COLS)
    dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][a]
    nr, nc = r + dr, c + dc
    # Si choca con pared, se queda
    if 0 <= nr < GW_ROWS and 0 <= nc < GW_COLS:
        s2 = nr * GW_COLS + nc
    else:
        s2 = s
    reward = GW_GOAL_R if s2 == GW_GOAL else GW_STEP_R
    return s2, reward

def gw_eps_greedy(Q, s, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(GW_ACTIONS))
    return int(np.argmax(Q[s]))

def gw_sarsa(n_episodes=500, alpha=0.5, gamma=0.9, eps=0.2, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((GW_N, GW_ACTIONS))
    returns = []
    for _ in range(n_episodes):
        s = 0
        a = gw_eps_greedy(Q, s, eps, rng)
        G = 0.0
        for _ in range(200):   # máximo 200 pasos por episodio
            s2, r = gw_step(s, a)
            G += r
            a2 = gw_eps_greedy(Q, s2, eps, rng) if s2 != GW_GOAL else 0
            q_next = Q[s2, a2] if s2 != GW_GOAL else 0.0
            Q[s, a] += alpha * (r + gamma * q_next - Q[s, a])
            s, a = s2, a2
            if s == GW_GOAL:
                break
        returns.append(G)
    return Q, returns

def gw_qlearning(n_episodes=500, alpha=0.5, gamma=0.9, eps=0.2, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((GW_N, GW_ACTIONS))
    returns = []
    for _ in range(n_episodes):
        s = 0
        G = 0.0
        for _ in range(200):
            a = gw_eps_greedy(Q, s, eps, rng)
            s2, r = gw_step(s, a)
            G += r
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
            s = s2
            if s == GW_GOAL:
                break
        returns.append(G)
    return Q, returns

print("Ambiente gridworld configurado.")
print(f"  Estados: {GW_N}, Acciones: {GW_ACTIONS}, Meta: {GW_GOAL}")

In [ ]:
Q_gw_sarsa, ret_gw_sarsa = gw_sarsa(n_episodes=1000, seed=42)
Q_gw_ql,    ret_gw_ql    = gw_qlearning(n_episodes=1000, seed=42)

print("Retorno promedio últimos 50 episodios:")
print(f"  SARSA:      {np.mean(ret_gw_sarsa[-50:]):.2f}")
print(f"  Q-learning: {np.mean(ret_gw_ql[-50:]):.2f}")

## 12. Visualizar la política aprendida en el gridworld

In [ ]:
ACTION_ARROWS = ['↑', '↓', '←', '→']

def plot_gw_policy(Q, title, ax):
    policy = np.argmax(Q, axis=1)
    values = np.max(Q, axis=1).reshape(GW_ROWS, GW_COLS)
    
    im = ax.imshow(values, cmap='YlOrRd', aspect='equal')
    
    for s in range(GW_N):
        r, c = divmod(s, GW_COLS)
        if s == GW_GOAL:
            ax.text(c, r, 'META\n+10', ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white')
        else:
            ax.text(c, r, ACTION_ARROWS[policy[s]], ha='center', va='center',
                    fontsize=16, color='black', fontweight='bold')
    
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=11)
    return im

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_gw_policy(Q_gw_sarsa, 'Política SARSA\n(gridworld 4×4)', axes[0])
plot_gw_policy(Q_gw_ql,    'Política Q-learning\n(gridworld 4×4)', axes[1])
plt.tight_layout()
plt.show()

## 13. Curvas de aprendizaje — gridworld

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
eps_gw = np.arange(1, 1001)

ax.plot(eps_gw[:len(smooth(ret_gw_sarsa))], smooth(ret_gw_sarsa, w=20),
        color=COLORS['blue'], lw=2, label='SARSA')
ax.plot(eps_gw[:len(smooth(ret_gw_ql))], smooth(ret_gw_ql, w=20),
        color=COLORS['orange'], lw=2, label='Q-learning')

ax.set_xlabel('Episodio')
ax.set_ylabel('Retorno')
ax.set_title('Curvas de aprendizaje — gridworld 4×4 (promedio móvil w=20)')
ax.legend()
plt.tight_layout()
plt.show()

## 14. Experimento: cambia los parámetros

Prueba distintas combinaciones de `alpha`, `gamma` y `eps` para la escalera y observa cómo cambia la convergencia.

In [ ]:
# ── Modifica estos valores ────────────────────────────────────────────────────
ALPHA_EXP   = 0.3   # tasa de aprendizaje
GAMMA_EXP   = 0.9   # descuento
EPSILON_EXP = 0.2   # exploración
# ─────────────────────────────────────────────────────────────────────────────

Q_s_exp, ret_s_exp, _ = sarsa(n_episodes=300, alpha=ALPHA_EXP,
                               gamma=GAMMA_EXP, eps=EPSILON_EXP, seed=42)
Q_q_exp, ret_q_exp, _ = q_learning(n_episodes=300, alpha=ALPHA_EXP,
                                    gamma=GAMMA_EXP, eps=EPSILON_EXP, seed=42)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smooth(ret_s_exp), color=COLORS['blue'], lw=2, label='SARSA')
ax.plot(smooth(ret_q_exp), color=COLORS['orange'], lw=2, label='Q-learning')
ax.set_xlabel('Episodio')
ax.set_ylabel('Retorno')
ax.set_title(f'α={ALPHA_EXP}, γ={GAMMA_EXP}, ε={EPSILON_EXP}')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nRetorno promedio últimos 20 eps:")
print(f"  SARSA: {np.mean(ret_s_exp[-20:]):.2f}")
print(f"  QL:    {np.mean(ret_q_exp[-20:]):.2f}")